# Pre-Deploy Camera Check

Capture a short **live video** from the robot's camera and display it inline.
Useful for visually inspecting the scene before deploying a policy.

**What it does:**
1. Connects to the robot via gRPC
2. Captures frames from `rgb_static` for a configurable duration
3. Encodes the frames as MP4 and displays the video inline
4. (Optional) Compares the last captured frame against a dataset's first frame

---
## 1. Configuration

In [ ]:
import pathlib

# Robot connection
SERVER_ENDPOINT = "localhost:50051"

# Capture settings
CAPTURE_DURATION_S = 5.0
CAPTURE_FPS = 15
IMAGE_WIDTH = 640
IMAGE_HEIGHT = 480

# Camera key (e.g. rgb_static, rgb_left, rgb_right)
CAMERA_KEY = "rgb_static"

# Optional: set to a local dataset path to compare live vs dataset
DATASET_PATH = None  # e.g. pathlib.Path("/data/lerobot/your_dataset_here")
EPISODE = None        # None = mean of all episodes, or an int for a specific episode

print(f"Server:   {SERVER_ENDPOINT}")
print(f"Camera:   {CAMERA_KEY}")
print(f"Capture:  {CAPTURE_DURATION_S}s @ {CAPTURE_FPS} FPS")
print(f"Resolution: {IMAGE_WIDTH}x{IMAGE_HEIGHT}")
print(f"Dataset:  {DATASET_PATH}")

---
## 2. Connect to Robot

In [ ]:
import grpc

from example_policies.robot_deploy.robot_io.robot_service import (
    robot_service_pb2,
    robot_service_pb2_grpc,
)

channel = grpc.insecure_channel(SERVER_ENDPOINT)
stub = robot_service_pb2_grpc.RobotServiceStub(channel)

# Test connection and list available cameras
test_resp = stub.GetState(robot_service_pb2.GetStateRequest())
available_cameras = list(test_resp.current_state.cameras.keys())
print(f"Connected to {SERVER_ENDPOINT}")
print(f"Available cameras: {available_cameras}")

---
## 3. Capture Live Video

In [ ]:
import time

import cv2
import numpy as np

from example_policies.data_ops.utils import image_processor

# Derive the ROS camera name from the camera key
side = CAMERA_KEY.replace("rgb_", "").replace("depth_", "")
is_depth = CAMERA_KEY.startswith("depth_")
modality = "depth" if is_depth else "color"
ros_cam_name = f"cam_{side}_{modality}_optical_frame"
print(f"ROS camera name: {ros_cam_name}")

if ros_cam_name not in available_cameras:
    raise RuntimeError(
        f"Camera '{ros_cam_name}' not found. Available: {available_cameras}"
    )

frames = []
interval = 1.0 / CAPTURE_FPS
total_frames = int(CAPTURE_DURATION_S * CAPTURE_FPS)

print(f"Capturing {total_frames} frames ({CAPTURE_DURATION_S}s @ {CAPTURE_FPS} FPS) ...")
t_start = time.time()

for i in range(total_frames):
    t_frame = time.time()

    resp = stub.GetState(robot_service_pb2.GetStateRequest())
    cam_data = resp.current_state.cameras[ros_cam_name]

    # process_image_bytes returns float32 [0,1] RGB
    img_f32 = image_processor.process_image_bytes(
        cam_data.data,
        width=IMAGE_WIDTH,
        height=IMAGE_HEIGHT,
        is_depth=is_depth,
    )
    # Convert to uint8 BGR for cv2.VideoWriter
    img_u8 = (img_f32 * 255).astype(np.uint8)
    img_bgr = cv2.cvtColor(img_u8, cv2.COLOR_RGB2BGR)
    frames.append(img_bgr)

    # Pace to target FPS
    elapsed = time.time() - t_frame
    if elapsed < interval:
        time.sleep(interval - elapsed)

t_end = time.time()
actual_fps = len(frames) / (t_end - t_start)
print(f"Captured {len(frames)} frames in {t_end - t_start:.1f}s (actual FPS: {actual_fps:.1f})")

---
## 4. Encode & Display Video

In [ ]:
import pathlib
import tempfile
from datetime import datetime

from IPython.display import Video, display

# Write frames to a temporary MP4
tmp_path = tempfile.mktemp(suffix=".mp4")
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(tmp_path, fourcc, actual_fps, (IMAGE_WIDTH, IMAGE_HEIGHT))

for frame in frames:
    writer.write(frame)
writer.release()

# Save a persistent copy
output_dir = pathlib.Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_path = output_dir / f"predeploy_camera_check_{timestamp}.mp4"

import shutil
shutil.copy2(tmp_path, save_path)
print(f"Saved video to {save_path}")

# Display inline
display(Video(str(save_path), embed=True))

---
## 5. (Optional) Dataset Comparison

If `DATASET_PATH` is set, compare the last captured live frame against the dataset's first frame.

In [ ]:
import matplotlib.pyplot as plt

if DATASET_PATH is not None:
    from example_policies.robot_deploy.pre_deploy_check import _load_dataset_first_frame

    # Load dataset first frame (uint8 RGB)
    ep_label = f"episode {EPISODE}" if EPISODE is not None else "mean of all episodes"
    print(f"Loading dataset first frame ({ep_label}) ...")
    dataset_img = _load_dataset_first_frame(DATASET_PATH, CAMERA_KEY, EPISODE)
    print(f"  Dataset image shape: {dataset_img.shape}")

    # Last captured frame: convert BGR -> RGB uint8
    live_img = cv2.cvtColor(frames[-1], cv2.COLOR_BGR2RGB)

    # Resize live frame to match dataset if needed
    dh, dw = dataset_img.shape[:2]
    if (live_img.shape[0], live_img.shape[1]) != (dh, dw):
        live_img = cv2.resize(live_img, (dw, dh))

    # Blended overlay
    blended = (0.5 * dataset_img.astype(np.float64) + 0.5 * live_img.astype(np.float64)).astype(np.uint8)

    # Plot side-by-side
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(dataset_img)
    axes[0].set_title(f"Dataset ({ep_label})", fontsize=12)
    axes[0].axis("off")

    axes[1].imshow(live_img)
    axes[1].set_title("Live Camera (last frame)", fontsize=12)
    axes[1].axis("off")

    axes[2].imshow(blended)
    axes[2].set_title("Overlay (50/50 blend)", fontsize=12)
    axes[2].axis("off")

    fig.suptitle(
        f"Pre-Deploy Comparison \u2014 {CAMERA_KEY}",
        fontsize=14,
        fontweight="bold",
    )
    plt.tight_layout()

    comparison_path = output_dir / f"predeploy_camera_comparison_{timestamp}.png"
    fig.savefig(comparison_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved comparison to {comparison_path}")
else:
    print("Skipping dataset comparison (DATASET_PATH is None).")

---
## 6. Cleanup

In [ ]:
channel.close()
print("gRPC channel closed. Done!")